In [1]:
import pandas as pd
import pyodbc

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=192.168.1.39;"
    "DATABASE=MIADWH;"
    "Trusted_Connection=yes;"
)

with pyodbc.connect(conn_str) as conn:
    df = pd.read_sql(
        """
        SELECT *
        FROM [MIADWH].[dbo].[View_test_Berlin_agg]
        """,
        conn
    )

print(df.head())

C:\Users\USER001381\AppData\Local\Temp\3\ipykernel_22892\4256774389.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


   year  Month_num             Pharma_Advertiser    Category Brand_AllGRPs  \
0  2020          1                      POLYSORB      Others        Others   
1  2020          1  BERLIN-CHEMIE MENARINI GROUP  FASTUM GEL    FASTUM GEL   
2  2020          1                          TEVA      Others        Others   
3  2020          1                     NEARMEDIC      Others        Others   
4  2020          1                        PFIZER      Others        Others   

   Cost_est_RUB_after_coeff      GRP_20  
0                      0.00    0.000000  
1                 191531.41    9.728936  
2               11321295.83  241.510937  
3                      0.00    0.000000  
4                 202234.22    4.716925  


In [2]:
result_full=df[['Pharma_Advertiser','year','Month_num','Cost_est_RUB_after_coeff','GRP_20']]
result_year=result_full.pivot_table(index='Pharma_Advertiser',columns='year',values=['GRP_20','Cost_est_RUB_after_coeff'],aggfunc='sum')

total = result_year.sum()
others = result_year.iloc[30:].sum()
result_year['Cost_est_RUB_after_coeff']

# добавляем строки
result_year.loc['Others'] = others
result_year.loc['TOTAL'] = total

cpp=result_year['Cost_est_RUB_after_coeff']/result_year['GRP_20']
cpp.columns = pd.MultiIndex.from_product([['CPP'], cpp.columns])
result_year = pd.concat([result_year, cpp], axis=1)
years = result_year['CPP'].columns

#считаем инфляцию growth 
for index, year in enumerate(years):
    if index == 0:
        continue
    prev_year = years[index - 1]
    result_year[('CPP_growth', year)] = (((result_year[('CPP', year)]/result_year[('CPP', prev_year)])-1)*100).round(0)

for index, year in enumerate(years):
    if index == 0:
        continue
    prev_year = years[index - 1]
    result_year[('Cost_growth', year)] = (((result_year[('Cost_est_RUB_after_coeff', year)]/result_year[('Cost_est_RUB_after_coeff', prev_year)])-1)*100).round(0)

for index, year in enumerate(years):
    if index == 0:
        continue
    prev_year = years[index - 1]
    result_year[('GRP_20_growth', year)] = (((result_year[('GRP_20', year)]/result_year[('GRP_20', prev_year)])-1)*100).round(0)



#Выводим проценты
for i in years:
    result_year[('Cost_Percent', i)]=((result_year[('Cost_est_RUB_after_coeff',i)]/result_year.loc['TOTAL',('Cost_est_RUB_after_coeff', i)])*100).round(0)

for i in years:
    result_year[('GRP_Percent', i)]=((result_year[('GRP_20',i)]/result_year.loc['TOTAL',('GRP_20', i)])*100).round(0)
    

#вырезаем нужные строки
result_years_final = pd.concat([
    result_year.iloc[:]
])


result_years_final


Cost_est_RUB_after_coeff                              \
year                                         2020          2021          2022   
Pharma_Advertiser                                                               
ABBOTT LABORATORIES S.A.             1.426605e+09  2.163417e+09  5.817958e+08   
ABBVIE                                        NaN           NaN           NaN   
ACINO                                         NaN  1.114340e+06           NaN   
ALCEA                                         NaN           NaN           NaN   
ALKOY                                         NaN           NaN           NaN   
...                                           ...           ...           ...   
ZELDIS                               5.801250e+06  5.876322e+06  3.323903e+07   
ZELENAYA DUBRAVA (ZD)                2.101703e+07  5.768597e+06  1.291187e+07   
ZENTIVA FARMA                        3.178519e+07  7.144790e+07  8.502925e+07   
Others                               2.707953e+10  3.138698e+10  3.070998e+10   
TOTAL                                3.289668e+10  3.864813e+10  3.337364e+10   

                                                                    \
year                              2023          2024          2025   
Pharma_Advertiser                                                    
ABBOTT LABORATORIES S.A.           NaN  8.502699e+08  1.624659e+09   
ABBVIE                             NaN           NaN           NaN   
ACINO                              NaN           NaN           NaN   
ALCEA                              NaN           NaN  2.162216e+07   
ALKOY                              NaN           NaN           NaN   
...                                ...           ...           ...   
ZELDIS                    1.260975e+05  3.061221e+07  5.760363e+07   
ZELENAYA DUBRAVA (ZD)     5.219827e+06  6.617699e+06           NaN   
ZENTIVA FARMA             5.406260e+07  8.716040e+07  1.754953e+08   
Others                    3.280913e+10  4.887796e+10  5.547667e+10   
TOTAL                     3.333772e+10  5.042804e+10  5.835285e+10   

                                               GRP_20                 \
year                              2026           2020           2021   
Pharma_Advertiser                                                      
ABBOTT LABORATORIES S.A.  1.046494e+09   21865.363606   27947.415152   
ABBVIE                    0.000000e+00            NaN            NaN   
ACINO                              NaN            NaN      15.396744   
ALCEA                     1.754579e+07            NaN            NaN   
ALKOY                     0.000000e+00            NaN            NaN   
...                                ...            ...            ...   
ZELDIS                    6.309728e+06     101.669550     116.011530   
ZELENAYA DUBRAVA (ZD)              NaN     515.452400     108.657621   
ZENTIVA FARMA             8.311121e+07     348.373014     661.157035   
Others                    3.031389e+10  508778.291710  491716.677233   
TOTAL                     3.169522e+10  617458.749277  603121.371416   

                                         ... Cost_Percent                \
year                               2022  ...         2024   2025   2026   
Pharma_Advertiser                        ...                              
ABBOTT LABORATORIES S.A.   10657.880004  ...          2.0    3.0    3.0   
ABBVIE                              NaN  ...          NaN    NaN    0.0   
ACINO                               NaN  ...          NaN    NaN    NaN   
ALCEA                               NaN  ...          NaN    0.0    0.0   
ALKOY                               NaN  ...          NaN    NaN    0.0   
...                                 ...  ...          ...    ...    ...   
ZELDIS                       341.906586  ...          0.0    0.0    0.0   
ZELENAYA DUBRAVA (ZD)        170.262417  ...          0.0    NaN    NaN   
ZENTIVA FARMA                988.946675  ...          0.0    0.0    0.0   


In [3]:
result_full_month=df[['Pharma_Advertiser','year','Month_num','Cost_est_RUB_after_coeff','GRP_20']]
result_full_month=result_full_month[(result_full_month['Month_num'].between(1,7)) & (result_full_month['year'].between(2025,2026))]
result_full_month=result_full_month.pivot_table(index='Pharma_Advertiser',columns='year',values=['GRP_20','Cost_est_RUB_after_coeff'],aggfunc='sum')

result_full_month=result_full_month.sort_values(by=[('GRP_20',2026),('GRP_20',2025)],ascending=False)

total = result_full_month.sum()
others = result_full_month.iloc[30:].sum()

# добавляем строки
result_full_month.loc['Others'] = others
result_full_month.loc['TOTAL'] = total
result_full_month['Cost_est_RUB_after_coeff']


cpp=result_full_month['Cost_est_RUB_after_coeff']/result_full_month['GRP_20']
cpp.columns = pd.MultiIndex.from_product([['CPP'], cpp.columns])
result_full_month = pd.concat([result_full_month, cpp], axis=1)
years = result_full_month['CPP'].columns

#считаем инфляцию growth 
for index, year in enumerate(years):
    if index == 0:
        continue
    prev_year = years[index - 1]
    result_full_month[('CPP_growth', year)] = (((result_full_month[('CPP', year)]/result_full_month[('CPP', prev_year)])-1)*100).round(0)

for index, year in enumerate(years):
    if index == 0:
        continue
    prev_year = years[index - 1]
    result_full_month[('Cost_growth', year)] = (((result_full_month[('Cost_est_RUB_after_coeff', year)]/result_full_month[('Cost_est_RUB_after_coeff', prev_year)])-1)*100).round(0)

for index, year in enumerate(years):
    if index == 0:
        continue
    prev_year = years[index - 1]
    result_full_month[('GRP_20_growth', year)] = (((result_full_month[('GRP_20', year)]/result_full_month[('GRP_20', prev_year)])-1)*100).round(0)


#Выводим проценты
for i in years:
    result_full_month[('Cost_Percent', i)]=((result_full_month[('Cost_est_RUB_after_coeff',i)]/result_full_month.loc['TOTAL',('Cost_est_RUB_after_coeff', i)])*100).round(0)

for i in years:
    result_full_month[('GRP_Percent', i)]=((result_full_month[('GRP_20',i)]/result_full_month.loc['TOTAL',('GRP_20', i)])*100).round(0)
    

#вырезаем нужные строки
result_month_final = pd.concat([
    result_full_month.iloc[:]
])


result_month_final.columns = pd.MultiIndex.from_tuples(
    [(metric, f'{year} 1-7') for metric, year in result_month_final.columns])

result_month_final.to_excel('month.xlsx')



In [4]:
final=result_month_final.merge(result_years_final,on='Pharma_Advertiser',how='left')

final.columns
#final.sort_values(('GRP_20','2026 1-7'),ascending=False)

final=pd.concat([final.iloc[:30],final.iloc[-2:]])

final['Cost_est_RUB_after_coeff']=(final['Cost_est_RUB_after_coeff']/1000000).round(0)

final




Cost_est_RUB_after_coeff                  GRP_20  \
                                            2025 1-7 2026 1-7       2025 1-7   
Pharma_Advertiser                                                              
OTCPHARM                                      4309.0   4629.0   37820.248252   
KLUB ZDOROVYA                                   91.0   1517.0    1710.116092   
EVALAR                                        2151.0   1386.0   28920.077896   
Binnopharm Group                              2451.0   2489.0   22466.175718   
NIZHPHARM                                     1173.0   1657.0   12556.479284   
PROMOMED                                       863.0   2062.0    5688.472092   
PETROVAKS                                     1077.0   1127.0   11179.288507   
SANOFI AVENTIS                                1124.0   1309.0    8713.332662   
TEVA                                          1501.0   1293.0   12026.615836   
ABBOTT LABORATORIES S.A.                       726.0   1046.0    6820.741918   
PHARMSTANDARD                                  692.0   1482.0    3577.962905   
DR. REDDY'S LABORATORIES                      1041.0   1527.0    6324.566795   
MATERIA MEDICA                                1505.0    713.0   13004.503276   
POLPHARMA                                      880.0    812.0    6493.189106   
NATSIONALNY TSENTR ZDOROVYA                    199.0    398.0    4167.646226   
VALENTA (HOLDING)                              435.0   1019.0    2672.137013   
KENVUE                                         262.0    711.0    1512.716191   
SERVIER                                        547.0    657.0    3512.083400   
HEEL RUS                                       289.0    300.0    2651.940486   
IP PALCHUNOVA E.S.                               NaN    155.0            NaN   
PFK OBNOVLENIE                                 541.0    418.0    3760.536560   
PROTEK                                         435.0    272.0    3820.865835   
EGIS                                           467.0    489.0    2661.791090   
POLYSAN                                        274.0    359.0    1892.735201   
BIONORICA ARZNEIMITTEL GMBH                    206.0    455.0    1152.432455   
KRKA D.D.                                      272.0    308.0    2160.659325   
INNOTECH INTERNACIONAL                         252.0    244.0    1992.720611   
TIBET HIMALAYA                                 139.0    155.0    2040.252650   
POZITIV BIO                                     58.0    102.0     992.065753   
DR.THEISS NATURWAREN RUS                       292.0    211.0    3198.417116   
Others                                        3190.0   2394.0   26782.177571   
TOTAL                                        27443.0  31695.0  242272.947822   

                                                      CPP                 \
                                  2026 1-7       2025 1-7       2026 1-7   
Pharma_Advertiser                                                          
OTCPHARM                      31249.714812  113920.509213  148133.971009   
KLUB ZDOROVYA                 18217.190632   53206.529767   83256.193191   
EVALAR                        18076.869763   74389.488322   76665.655069   
Binnopharm Group              17750.243319  109081.953821  140231.586395   
NIZHPHARM                     13175.088257   93448.279536  125766.046285   
PROMOMED                      10682.280697  151749.382900  192984.638034   
PETROVAKS                      8614.728115   96348.397317  130778.383430   
SANOFI AVENTIS                 8458.174110  129036.077692  154799.857661   
TEVA                           8246.173487  124843.092492  156792.383736   
ABBOTT LABORATORIES S.A.       7715.169902  106380.332959  135641.101582   
PHARMSTANDARD                  7090.499466  193485.860824  208984.233656   
DR. REDDY'S LABORATORIES       6945.987675  164600.738867  219855.279230   
MATERIA MEDICA                 5622.372132  115707.940948  126799.139522   
POLPHARMA  

In [5]:
def highlight_rows(row):
    if row.name == 'TOTAL':
        return [
            'background-color: darkred; color: white; font-weight: bold'
        ] * len(row)

    elif row.name == 'Others':
        return [
            'background-color: #f4cccc; font-weight: bold'
        ] * len(row)

    return [''] * len(row)


styled = (
    final.style
    .apply(highlight_rows, axis=1)
    .set_table_styles([
        {
            'selector': 'th.row_heading',
            'props': [
                ('font-weight', 'bold'),
                ('font-style', 'italic')
            ]
        },
        {
            'selector': 'th.index_name',
            'props': [
                ('font-weight', 'bold'),
                ('font-style', 'italic')
            ]
        }
    ])
)

styled.to_excel('final.xlsx')